# Chapter 1 Computational Lab
## Combinatorial Foundations for Probability

This interactive notebook accompanies Chapter 1 of *Probability Theory with Python and AI*.  
Its purpose is not to replace proofs by computation, but to make the **structure of the sample space visible** before a counting formula is used.

### Learning goals

By the end of the lab you should be able to:

1. work with the basic set operations used throughout probability;
2. recognize the addition, multiplication and division principles;
3. use the generalized pigeonhole principle;
4. distinguish factorials, falling factorials, permutations and combinations;
5. count arrangements with repeated types and circular arrangements;
6. use binomial/multinomial coefficients, stars and bars, and inclusion--exclusion;
7. choose correctly among the four standard sampling schemes;
8. turn counting results into finite uniform probabilities;
9. verify the hypergeometric formula and Galileo's three-dice calculation;
10. audit an AI-generated combinatorial solution rather than accepting a formula blindly.

> **Working rule.** Before moving a slider or pressing a button, write down what one elementary outcome is and predict which counting principle should apply.


## 0. Notebook setup

The setup cell imports only standard scientific-Python tools plus `ipywidgets`.  
All combinatorial formulas used later are implemented explicitly so that they can also be tested automatically.


In [ ]:
import math
import random
from itertools import product, combinations, permutations

import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import HTML, Math, Markdown, clear_output, display

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass


def show_math_result(title, *latex_lines, note=None):
    display(HTML(
        f"<div style='border-left:5px solid;padding:8px 12px;"
        f"margin:8px 0'><b>{title}</b></div>"
    ))
    for line in latex_lines:
        display(Math(line))
    if note:
        display(Markdown(note))


def falling_factorial(n, k):
    # Falling factorial (n)_k, with value 0 when k>n.
    if n < 0 or k < 0:
        raise ValueError("n and k must be non-negative.")
    if k > n:
        return 0
    return math.factorial(n) // math.factorial(n - k)


def multinomial(counts):
    if any(c < 0 for c in counts):
        raise ValueError("Counts must be non-negative.")
    total = sum(counts)
    value = math.factorial(total)
    for c in counts:
        value //= math.factorial(c)
    return value


def derangement(n):
    if n < 0:
        raise ValueError("n must be non-negative.")
    if n == 0:
        return 1
    if n == 1:
        return 0
    d0, d1 = 1, 0
    for k in range(2, n + 1):
        d0, d1 = d1, (k - 1) * (d1 + d0)
    return d1


def onto_count(n, m):
    if n < 0 or m < 0:
        raise ValueError("n and m must be non-negative.")
    if m == 0:
        return int(n == 0)
    return sum(
        (-1) ** j * math.comb(m, j) * (m - j) ** n
        for j in range(m + 1)
    )


def birthday_collision_probability(k, n=365):
    if k < 0 or n <= 0:
        raise ValueError
    if k > n:
        return 1.0
    no_collision = 1.0
    for j in range(k):
        no_collision *= (n - j) / n
    return 1.0 - no_collision


def parse_int_set(text):
    text = text.strip()
    if not text:
        return set()
    return {int(item.strip()) for item in text.split(",") if item.strip()}


display(HTML(
    "<div style='padding:10px;border:1px solid'>"
    "<b>Setup complete.</b> Run the notebook from top to bottom; "
    "all interactive controls are now available.</div>"
))


## 1. Finite sets and the language of counting

For subsets $A,B\subseteq\Omega$,

$$
A\cup B,\qquad A\cap B,\qquad A\setminus B,\qquad A^c=\Omega\setminus A
$$

denote union, intersection, difference and complement. The **symmetric difference**

$$
A\triangle B
=(A\setminus B)\cup(B\setminus A)
=(A\cup B)\setminus(A\cap B)
$$

contains the elements that belong to **exactly one** of $A$ and $B$.

The first panel computes these sets exactly. The second is a schematic Venn-style picture of the selected operation.


In [ ]:
universe_text = widgets.Text(value="1,2,3,4,5,6,7,8", description="Ω")
set_a_text = widgets.Text(value="1,2,3,5", description="A")
set_b_text = widgets.Text(value="3,4,5,7", description="B")
set_operation = widgets.Dropdown(
    options=[
        ("Union  A ∪ B", "union"),
        ("Intersection  A ∩ B", "intersection"),
        ("Difference  A \\ B", "difference"),
        ("Complement  Aᶜ", "complement"),
        ("Symmetric difference  A △ B", "symmetric"),
    ],
    value="symmetric",
    description="Operation",
)
set_output = widgets.Output()


def set_operation_value(U, A, B, op):
    if op == "union":
        return A | B
    if op == "intersection":
        return A & B
    if op == "difference":
        return A - B
    if op == "complement":
        return U - A
    return A ^ B


def draw_set_operation(op):
    x = np.linspace(-2.2, 2.2, 320)
    y = np.linspace(-1.5, 1.5, 240)
    Xg, Yg = np.meshgrid(x, y)
    Am = (Xg + 0.65) ** 2 + Yg ** 2 <= 1.0
    Bm = (Xg - 0.65) ** 2 + Yg ** 2 <= 1.0

    if op == "union":
        mask, title = Am | Bm, r"$A\cup B$"
    elif op == "intersection":
        mask, title = Am & Bm, r"$A\cap B$"
    elif op == "difference":
        mask, title = Am & ~Bm, r"$A\setminus B$"
    elif op == "complement":
        mask, title = ~Am, r"$A^c$"
    else:
        mask, title = Am ^ Bm, r"$A\triangle B$"

    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    ax.contourf(Xg, Yg, mask.astype(int), levels=[0.5, 1.5], alpha=0.28)
    theta = np.linspace(0, 2*np.pi, 400)
    ax.plot(-0.65 + np.cos(theta), np.sin(theta))
    ax.plot(0.65 + np.cos(theta), np.sin(theta))
    ax.plot([-2.15, 2.15, 2.15, -2.15, -2.15],
            [-1.4, -1.4, 1.4, 1.4, -1.4])
    ax.text(-1.1, 0.85, "A")
    ax.text(1.05, 0.85, "B")
    ax.text(-2.02, 1.18, r"$\Omega$")
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()


def update_sets(*_):
    with set_output:
        clear_output(wait=True)
        try:
            U = parse_int_set(universe_text.value)
            A = parse_int_set(set_a_text.value)
            B = parse_int_set(set_b_text.value)
        except ValueError:
            display(Markdown("**Use comma-separated integers.**"))
            return

        if not A <= U or not B <= U:
            display(Markdown("**Both $A$ and $B$ must be subsets of $Ω$.**"))
            return

        result = set_operation_value(U, A, B, set_operation.value)
        display(Markdown(
            f"**Exact set result:** `{sorted(result)}`  \n"
            f"$|A|={len(A)}, |B|={len(B)}, |A\\cap B|={len(A&B)}$"
        ))
        if set_operation.value == "symmetric":
            display(Math(
                rf"|A\triangle B|=|A|+|B|-2|A\cap B|"
                rf"={len(A)}+{len(B)}-2({len(A&B)})={len(result)}"
            ))
        draw_set_operation(set_operation.value)


for control in (universe_text, set_a_text, set_b_text, set_operation):
    control.observe(update_sets, names="value")

display(widgets.VBox([
    widgets.HBox([universe_text, set_operation]),
    widgets.HBox([set_a_text, set_b_text]),
    set_output,
]))
update_sets()


### Power sets and cardinality

For a finite set $\Omega$ with $n$ elements, the power set

$$
\mathcal P(\Omega)=\{A:A\subseteq\Omega\}
$$

has $2^n$ elements: each element of $\Omega$ independently has two statuses, **in** or **out**.


In [ ]:
power_n = widgets.IntSlider(value=4, min=0, max=10, description="n")
power_output = widgets.Output()

def update_power(*_):
    with power_output:
        clear_output(wait=True)
        n = power_n.value
        subsets = []
        if n <= 6:
            base = list(range(1, n + 1))
            for r in range(n + 1):
                subsets.extend(combinations(base, r))
        show_math_result(
            "Power-set count",
            rf"|\mathcal P(\Omega)|=2^{{|\Omega|}}=2^{{{n}}}={2**n}",
            note=("For $n≤6$, the subsets are listed below." if n <= 6
                  else "The list is suppressed for readability.")
        )
        if n <= 6:
            display(subsets)

power_n.observe(update_power, names="value")
display(widgets.VBox([power_n, power_output]))
update_power()


## 2. The three fundamental counting principles

The chapter repeatedly uses three structural rules.

- **Addition principle:** disjoint alternatives add.
- **Multiplication principle:** successive choices multiply.
- **Division principle:** if every visible object has been counted exactly $q$ times, divide the labelled count by $q$.

These are principles about **how outcomes are represented**, not formulas to memorize in isolation.


In [ ]:
add_a = widgets.IntSlider(value=7, min=0, max=30, description="|A|")
add_b = widgets.IntSlider(value=5, min=0, max=30, description="|B|")
stages_text = widgets.Text(value="4,3,2", description="Stages")
labelled_count = widgets.IntText(value=720, description="Labelled")
overcount = widgets.IntText(value=6, description="Each seen")
principles_output = widgets.Output()

def update_principles(*_):
    with principles_output:
        clear_output(wait=True)
        display(Math(
            rf"\text{{Addition (disjoint): }}\quad |A\cup B|={add_a.value}+{add_b.value}={add_a.value+add_b.value}"
        ))
        try:
            stages = [int(x.strip()) for x in stages_text.value.split(",") if x.strip()]
            if not stages or any(x < 0 for x in stages):
                raise ValueError
            prod_value = math.prod(stages)
            prod_latex = r"\cdot".join(map(str, stages))
            display(Math(rf"\text{{Multiplication: }}\quad {prod_latex}={prod_value}"))
        except ValueError:
            display(Markdown("**Stages must be comma-separated non-negative integers.**"))

        if overcount.value <= 0:
            display(Markdown("**The overcount factor must be positive.**"))
        elif labelled_count.value % overcount.value == 0:
            display(Math(
                rf"\text{{Division: }}\quad \frac{{{labelled_count.value}}}{{{overcount.value}}}"
                rf"={labelled_count.value // overcount.value}"
            ))
        else:
            display(Math(
                rf"\text{{Division: }}\quad \frac{{{labelled_count.value}}}{{{overcount.value}}}"
                rf"={labelled_count.value / overcount.value:.4f}"
            ))

for control in (add_a, add_b, stages_text, labelled_count, overcount):
    control.observe(update_principles, names="value")

display(widgets.VBox([
    widgets.HTML("<b>Addition principle</b>"),
    widgets.HBox([add_a, add_b]),
    widgets.HTML("<b>Multiplication principle</b>"),
    stages_text,
    widgets.HTML("<b>Division principle</b>"),
    widgets.HBox([labelled_count, overcount]),
    principles_output,
]))
update_principles()


## 3. Generalized pigeonhole principle

If $N$ objects are placed into $m$ boxes, then some box contains at least

$$
\left\lceil\frac Nm\right\rceil
$$

objects.

This is an **existence statement**: it guarantees concentration without telling us which box achieves it.


In [ ]:
pigeon_N = widgets.IntSlider(value=23, min=0, max=100, description="objects N")
pigeon_m = widgets.IntSlider(value=5, min=1, max=20, description="boxes m")
pigeon_output = widgets.Output()

def update_pigeon(*_):
    with pigeon_output:
        clear_output(wait=True)
        N, m = pigeon_N.value, pigeon_m.value
        guaranteed = math.ceil(N / m)
        q, r = divmod(N, m)
        balanced = [q + 1] * r + [q] * (m - r)
        display(Math(
            rf"\left\lceil\frac{{N}}{{m}}\right\rceil"
            rf"=\left\lceil\frac{{{N}}}{{{m}}}\right\rceil={guaranteed}"
        ))
        display(Markdown(
            f"A maximally balanced allocation is `{balanced}`. "
            f"Even in this best-spread case, a box contains {guaranteed} object(s)."
        ))

for c in (pigeon_N, pigeon_m):
    c.observe(update_pigeon, names="value")
display(widgets.VBox([widgets.HBox([pigeon_N, pigeon_m]), pigeon_output]))
update_pigeon()


## 4. Factorials, falling factorials and ordered selections

The number of linear orderings of $n$ distinct objects is

$$
n!=n(n-1)\cdots2\cdot1.
$$

For an ordered sample of $k$ **distinct** objects from $n$,

$$
(n)_k=n(n-1)\cdots(n-k+1)=\frac{n!}{(n-k)!}.
$$

This is the **falling factorial**. With replacement, each of the $k$ positions again has $n$ choices, so the count is $n^k$.


In [ ]:
n_ordered = widgets.IntSlider(value=8, min=1, max=20, description="n")
k_ordered = widgets.IntSlider(value=3, min=0, max=20, description="k")
replacement_ordered = widgets.Checkbox(value=False, description="With replacement")
ordered_output = widgets.Output()

def update_ordered(*_):
    with ordered_output:
        clear_output(wait=True)
        n, k = n_ordered.value, k_ordered.value
        if replacement_ordered.value:
            value = n ** k
            show_math_result(
                "Ordered sampling with replacement",
                rf"n^k={n}^{{{k}}}={value:,}",
                note="Each position may use any of the available types."
            )
        elif k > n:
            show_math_result(
                "Ordered sampling without replacement",
                rf"k={k}>n={n}\Longrightarrow (n)_k=0",
                note="There are not enough distinct objects to fill all positions."
            )
        else:
            value = falling_factorial(n, k)
            show_math_result(
                "Ordered sampling without replacement",
                rf"(n)_k=\frac{{n!}}{{(n-k)!}}"
                rf"=\frac{{{n}!}}{{({n}-{k})!}}={value:,}"
            )

for c in (n_ordered, k_ordered, replacement_ordered):
    c.observe(update_ordered, names="value")
display(widgets.VBox([
    widgets.HBox([n_ordered, k_ordered]),
    replacement_ordered,
    ordered_output,
]))
update_ordered()


## 5. Unordered selections and binomial coefficients

An unordered selection of $k$ distinct objects from $n$ is a $k$-element subset. Its count is

$$
\binom nk=\frac{n!}{k!(n-k)!}.
$$

The factor $k!$ appears because each unordered selection is represented by $k!$ orderings.

Pascal's identity,

$$
\binom{n+1}{k}=\binom nk+\binom n{k-1},
$$

comes from splitting $k$-subsets according to whether they contain one fixed element.


In [ ]:
comb_n = widgets.IntSlider(value=10, min=1, max=30, description="n")
comb_k = widgets.IntSlider(value=4, min=0, max=30, description="k")
pascal_output = widgets.Output()

def update_combinations(*_):
    with pascal_output:
        clear_output(wait=True)
        n, k = comb_n.value, comb_k.value
        if k > n:
            display(Math(rf"k={k}>n={n}\Longrightarrow \binom{{n}}{{k}}=0"))
            return

        value = math.comb(n, k)
        ordered = falling_factorial(n, k)
        display(Math(
            rf"\binom{{{n}}}{{{k}}}=\frac{{({n})_{{{k}}}}}{{{k}!}}"
            rf"=\frac{{{ordered}}}{{{math.factorial(k)}}}={value}"
        ))

        row = [math.comb(n, j) for j in range(n + 1)]
        fig, ax = plt.subplots(figsize=(8, 3.1))
        ax.bar(range(n + 1), row)
        ax.set_title(rf"Row $n={n}$ of Pascal's triangle")
        ax.set_xlabel(r"$j$")
        ax.set_ylabel(r"$\binom{n}{j}$")
        ax.set_xticks(range(n + 1))
        plt.show()

        display(Math(rf"\sum_{{j=0}}^{{{n}}}\binom{{{n}}}{{j}}=2^{{{n}}}={2**n:,}"))
        if 1 <= k <= n:
            display(Math(
                rf"\binom{{{n+1}}}{{{k}}}"
                rf"=\binom{{{n}}}{{{k}}}+\binom{{{n}}}{{{k-1}}}"
                rf"={math.comb(n,k)}+{math.comb(n,k-1)}"
                rf"={math.comb(n+1,k)}"
            ))

for c in (comb_n, comb_k):
    c.observe(update_combinations, names="value")
display(widgets.VBox([widgets.HBox([comb_n, comb_k]), pascal_output]))
update_combinations()


## 6. Repeated types, multinomial counting and circular permutations

If $n$ positions contain $m$ types with counts $n_1,\ldots,n_m$, then

$$
\binom{n}{n_1,\ldots,n_m}
=\frac{n!}{n_1!\cdots n_m!}.
$$

For $n\ge2$ distinct objects arranged around a circle, rotations represent the same arrangement, so

$$
\frac{n!}{n}=(n-1)!.
$$


In [ ]:
multinomial_input = widgets.Text(value="5,3,2", description="Type counts")
circular_n = widgets.IntSlider(value=6, min=2, max=15, description="circle n")
arrangement_output = widgets.Output()

def update_arrangements(*_):
    with arrangement_output:
        clear_output(wait=True)
        try:
            counts = [int(x.strip()) for x in multinomial_input.value.split(",") if x.strip()]
            if not counts or any(x < 0 for x in counts):
                raise ValueError
        except ValueError:
            display(Markdown("**Enter non-negative integer type counts separated by commas.**"))
            return

        n = sum(counts)
        value = multinomial(counts)
        denominator = r"\,".join(f"{x}!" for x in counts)
        display(Math(rf"\frac{{{n}!}}{{{denominator}}}={value:,}"))

        c = circular_n.value
        display(Math(
            rf"\text{{Circular arrangements of {c} distinct objects}}"
            rf"=({c}-1)!={math.factorial(c-1):,}"
        ))

for c in (multinomial_input, circular_n):
    c.observe(update_arrangements, names="value")
display(widgets.VBox([multinomial_input, circular_n, arrangement_output]))
update_arrangements()


## 7. Stars and bars

The number of non-negative integer solutions of

$$
x_1+\cdots+x_m=r
$$

is

$$
\binom{r+m-1}{m-1}.
$$

If all variables must be positive and $r\ge m$, subtract one from each variable first; the count becomes

$$
\binom{r-1}{m-1}.
$$

For small examples the notebook lists every solution so that the bijection is visible.


In [ ]:
stars_r = widgets.IntSlider(value=7, min=0, max=25, description="r")
stars_m = widgets.IntSlider(value=3, min=1, max=7, description="m")
stars_positive = widgets.Checkbox(value=False, description="Require positive values")
stars_output = widgets.Output()

def compositions(total, parts, minimum=0):
    remaining = total - parts * minimum
    if remaining < 0:
        return []

    result = []
    def build(prefix, left, slots):
        if slots == 1:
            result.append(tuple(prefix + [left + minimum]))
            return
        for value in range(left + 1):
            build(prefix + [value + minimum], left - value, slots - 1)
    build([], remaining, parts)
    return result

def update_stars(*_):
    with stars_output:
        clear_output(wait=True)
        r, m = stars_r.value, stars_m.value
        minimum = 1 if stars_positive.value else 0

        if minimum == 0:
            count = math.comb(r + m - 1, m - 1)
            formula = rf"\binom{{r+m-1}}{{m-1}}=\binom{{{r+m-1}}}{{{m-1}}}={count}"
        elif r >= m:
            count = math.comb(r - 1, m - 1)
            formula = rf"\binom{{r-1}}{{m-1}}=\binom{{{r-1}}}{{{m-1}}}={count}"
        else:
            count = 0
            formula = rf"r={r}<m={m}\Longrightarrow 0"

        display(Math(formula))
        sols = compositions(r, m, minimum)
        if len(sols) <= 60:
            display(Markdown(f"**All {len(sols)} solutions:** `{sols}`"))
        else:
            display(Markdown(f"**There are {len(sols)} solutions.** The full list is suppressed."))

for c in (stars_r, stars_m, stars_positive):
    c.observe(update_stars, names="value")
display(widgets.VBox([
    widgets.HBox([stars_r, stars_m]),
    stars_positive,
    stars_output,
]))
update_stars()


## 8. Inclusion--exclusion

For three finite sets,

$$
|A\cup B\cup C|
=|A|+|B|+|C|
-|A\cap B|-|A\cap C|-|B\cap C|
+|A\cap B\cap C|.
$$

Inclusion--exclusion repairs the overcount created when an object satisfies more than one condition.


In [ ]:
ie_controls = {
    "U": widgets.IntSlider(value=100, min=1, max=200, description="|Ω|"),
    "A": widgets.IntSlider(value=35, min=0, max=100, description="|A|"),
    "B": widgets.IntSlider(value=28, min=0, max=100, description="|B|"),
    "C": widgets.IntSlider(value=20, min=0, max=100, description="|C|"),
    "AB": widgets.IntSlider(value=9, min=0, max=100, description="|A∩B|"),
    "AC": widgets.IntSlider(value=6, min=0, max=100, description="|A∩C|"),
    "BC": widgets.IntSlider(value=5, min=0, max=100, description="|B∩C|"),
    "ABC": widgets.IntSlider(value=2, min=0, max=100, description="|A∩B∩C|"),
}
ie_output = widgets.Output()

def update_ie(*_):
    with ie_output:
        clear_output(wait=True)
        v = {name: ctrl.value for name, ctrl in ie_controls.items()}
        union = v["A"] + v["B"] + v["C"] - v["AB"] - v["AC"] - v["BC"] + v["ABC"]
        neither = v["U"] - union

        issues = []
        if v["A"] > v["U"] or v["B"] > v["U"] or v["C"] > v["U"]:
            issues.append("A set cannot be larger than the universe.")
        if v["AB"] > min(v["A"], v["B"]):
            issues.append("|A∩B| is too large.")
        if v["AC"] > min(v["A"], v["C"]):
            issues.append("|A∩C| is too large.")
        if v["BC"] > min(v["B"], v["C"]):
            issues.append("|B∩C| is too large.")
        if v["ABC"] > min(v["AB"], v["AC"], v["BC"]):
            issues.append("The triple intersection is too large.")
        if union < 0 or union > v["U"]:
            issues.append("The supplied counts imply an impossible union size.")

        if issues:
            display(Markdown("**Consistency warning:** " + " ".join(issues)))
        display(Math(
            rf"|A\cup B\cup C|={v['A']}+{v['B']}+{v['C']}"
            rf"-{v['AB']}-{v['AC']}-{v['BC']}+{v['ABC']}={union}"
        ))
        display(Math(rf"|\Omega\setminus(A\cup B\cup C)|={v['U']}-{union}={neither}"))

for ctrl in ie_controls.values():
    ctrl.observe(update_ie, names="value")
display(widgets.VBox([
    widgets.HBox([ie_controls["U"], ie_controls["A"], ie_controls["B"], ie_controls["C"]]),
    widgets.HBox([ie_controls["AB"], ie_controls["AC"], ie_controls["BC"], ie_controls["ABC"]]),
    ie_output,
]))
update_ie()


### Derangements and onto functions

Two important inclusion--exclusion applications are

$$
D_n=n!\sum_{j=0}^{n}\frac{(-1)^j}{j!},
$$

for derangements, and

$$
\sum_{j=0}^{m}(-1)^j\binom mj(m-j)^n
$$

for onto functions from an $n$-element domain to an $m$-element target.


In [ ]:
der_n = widgets.IntSlider(value=6, min=0, max=15, description="derangement n")
onto_n = widgets.IntSlider(value=6, min=0, max=15, description="domain n")
onto_m = widgets.IntSlider(value=3, min=1, max=10, description="target m")
inc_apps_output = widgets.Output()

def update_inc_apps(*_):
    with inc_apps_output:
        clear_output(wait=True)
        n = der_n.value
        display(Math(
            rf"D_{{{n}}}={derangement(n):,},\qquad "
            rf"\frac{{D_{{{n}}}}}{{{n}!}}={derangement(n)/math.factorial(n):.6f}"
        ))
        n2, m2 = onto_n.value, onto_m.value
        display(Math(
            rf"N_{{\mathrm{{onto}}}}({n2},{m2})"
            rf"=\sum_{{j=0}}^{{{m2}}}(-1)^j\binom{{{m2}}}{{j}}({m2}-j)^{{{n2}}}"
            rf"={onto_count(n2,m2):,}"
        ))

for c in (der_n, onto_n, onto_m):
    c.observe(update_inc_apps, names="value")
display(widgets.VBox([
    widgets.HBox([der_n, onto_n, onto_m]),
    inc_apps_output,
]))
update_inc_apps()


## 9. The four standard sampling schemes

Before selecting a formula, ask two questions:

1. **Is order recorded?**
2. **Is replacement allowed?**

$$
\begin{array}{c|c|c}
&\text{with replacement}&\text{without replacement}\\ \hline
\text{ordered}&n^k&(n)_k\\
\text{unordered}&\displaystyle\binom{n+k-1}{k}&\displaystyle\binom nk
\end{array}
$$

For the unordered-with-replacement case, $n$ refers to **types** and the outcome records multiplicities.


In [ ]:
scheme_order = widgets.Dropdown(
    options=[("Ordered", True), ("Unordered", False)],
    value=True,
    description="Order",
)
scheme_replacement = widgets.Dropdown(
    options=[("With replacement", True), ("Without replacement", False)],
    value=True,
    description="Reuse",
)
scheme_n = widgets.IntSlider(value=5, min=1, max=25, description="n")
scheme_k = widgets.IntSlider(value=3, min=0, max=25, description="k")
scheme_output = widgets.Output()

def update_scheme(*_):
    with scheme_output:
        clear_output(wait=True)
        ordered = scheme_order.value
        replace = scheme_replacement.value
        n, k = scheme_n.value, scheme_k.value

        if ordered and replace:
            value = n ** k
            formula = rf"n^k={n}^{{{k}}}={value}"
            description = "ordered sequences; types may repeat"
        elif ordered and not replace:
            value = falling_factorial(n, k)
            formula = rf"(n)_k={value}"
            description = "ordered selections of distinct objects"
        elif not ordered and replace:
            value = math.comb(n + k - 1, k)
            formula = rf"\binom{{n+k-1}}{{k}}=\binom{{{n+k-1}}}{{{k}}}={value}"
            description = "multisets / type-count vectors"
        else:
            value = math.comb(n, k) if k <= n else 0
            formula = rf"\binom{{n}}{{k}}={value}"
            description = "unordered subsets of distinct objects"

        display(Math(formula))
        display(Markdown(f"**Outcome interpretation:** {description}."))

        if n <= 5 and k <= 4:
            if ordered and replace:
                enum_count = len(list(product(range(n), repeat=k)))
            elif ordered and not replace:
                enum_count = len(list(permutations(range(n), k))) if k <= n else 0
            elif not ordered and replace:
                # Each multiset is equivalent to a non-negative count vector.
                enum_count = math.comb(n + k - 1, k)
            else:
                enum_count = len(list(combinations(range(n), k))) if k <= n else 0
            display(Markdown(f"Small-case enumeration check: **{enum_count}** outcomes."))

for c in (scheme_order, scheme_replacement, scheme_n, scheme_k):
    c.observe(update_scheme, names="value")
display(widgets.VBox([
    widgets.HBox([scheme_order, scheme_replacement]),
    widgets.HBox([scheme_n, scheme_k]),
    scheme_output,
]))
update_scheme()


## 10. Bridge to finite probability

If a finite sample space $\Omega$ is uniform, then

$$
\mathbb P(A)=\frac{|A|}{|\Omega|}.
$$

The assumption of equally likely elementary outcomes is essential. Counting alone does **not** assign probabilities when the outcomes have unequal weights.


In [ ]:
uniform_total = widgets.IntSlider(value=20, min=1, max=200, description="|Ω|")
uniform_event = widgets.IntSlider(value=7, min=0, max=200, description="|A|")
uniform_output = widgets.Output()

def update_uniform(*_):
    with uniform_output:
        clear_output(wait=True)
        total, event = uniform_total.value, uniform_event.value
        if event > total:
            display(Markdown("**An event cannot contain more outcomes than $Ω$.**"))
            return
        display(Math(
            rf"\mathbb P(A)=\frac{{|A|}}{{|\Omega|}}"
            rf"=\frac{{{event}}}{{{total}}}={event/total:.6f}"
        ))

for c in (uniform_total, uniform_event):
    c.observe(update_uniform, names="value")
display(widgets.VBox([widgets.HBox([uniform_total, uniform_event]), uniform_output]))
update_uniform()


### Hypergeometric counting

A population has $N$ distinct objects, $K$ of a specified type, and we draw $n$ objects uniformly without replacement. Then

$$
\mathbb P(X=k)
=
\frac{\binom Kk\binom{N-K}{n-k}}{\binom Nn}.
$$

This formula is a direct application of **favourable outcomes / total outcomes**.


In [ ]:
hyper_N = widgets.IntSlider(value=100, min=1, max=500, description="N")
hyper_K = widgets.IntSlider(value=20, min=0, max=200, description="K")
hyper_n = widgets.IntSlider(value=10, min=0, max=100, description="sample n")
hyper_k = widgets.IntSlider(value=2, min=0, max=100, description="k")
hyper_output = widgets.Output()

def update_hyper(*_):
    with hyper_output:
        clear_output(wait=True)
        N, K, n, k = hyper_N.value, hyper_K.value, hyper_n.value, hyper_k.value
        if K > N or n > N:
            display(Markdown("**Require $K≤N$ and $n≤N$.**"))
            return
        lower = max(0, n - (N - K))
        upper = min(n, K)
        if not (lower <= k <= upper):
            display(Math(
                rf"k={k}\notin\{{{lower},\ldots,{upper}\}}\Longrightarrow \mathbb P(X=k)=0"
            ))
            return
        fav = math.comb(K, k) * math.comb(N - K, n - k)
        total = math.comb(N, n)
        p = fav / total
        display(Math(
            rf"\mathbb P(X={k})"
            rf"=\frac{{\binom{{{K}}}{{{k}}}\binom{{{N-K}}}{{{n-k}}}}}"
            rf"{{\binom{{{N}}}{{{n}}}}}"
            rf"=\frac{{{fav}}}{{{total}}}\approx {p:.6f}"
        ))
        display(Markdown(f"Feasible support: **{lower}, …, {upper}**."))

for c in (hyper_N, hyper_K, hyper_n, hyper_k):
    c.observe(update_hyper, names="value")
display(widgets.VBox([
    widgets.HBox([hyper_N, hyper_K]),
    widgets.HBox([hyper_n, hyper_k]),
    hyper_output,
]))
update_hyper()


## 11. Historical problem: Galileo's three dice

Three fair dice have $6^3=216$ equally likely **ordered triples**.

Galileo's puzzle asks why a sum of $10$ occurs more often than a sum of $9$, even though both totals have six unordered representations. The answer is that those unordered representations do **not** all have the same number of orderings.

The code below enumerates the actual sample space.


In [ ]:
galileo_output = widgets.Output()

def show_galileo():
    with galileo_output:
        clear_output(wait=True)
        outcomes = list(product(range(1, 7), repeat=3))
        counts = {s: 0 for s in range(3, 19)}
        for w in outcomes:
            counts[sum(w)] += 1

        display(Math(
            rf"N_9={counts[9]},\qquad N_{{10}}={counts[10]},\qquad |\Omega|={len(outcomes)}"
        ))
        display(Math(
            rf"\mathbb P(S=9)=\frac{{25}}{{216}}\approx {counts[9]/216:.6f},\qquad"
            rf"\mathbb P(S=10)=\frac{{27}}{{216}}=\frac18"
        ))
        display(Markdown(
            "**Conclusion:** the elementary outcomes are ordered triples. "
            "Counting only unordered decompositions hides their different multiplicities."
        ))

        xs = list(counts)
        ys = [counts[s] for s in xs]
        fig, ax = plt.subplots(figsize=(8, 3.3))
        ax.bar(xs, ys)
        ax.set_xlabel("sum of three dice")
        ax.set_ylabel("number of ordered triples")
        ax.set_xticks(xs)
        ax.annotate("25", (9, counts[9]), xytext=(9, counts[9] + 2), ha="center")
        ax.annotate("27", (10, counts[10]), xytext=(10, counts[10] + 2), ha="center")
        plt.show()

display(galileo_output)
show_galileo()


## 12. Birthday collisions

With $k$ labelled individuals and $n$ equally likely dates, there are $n^k$ ordered date sequences. If $k\le n$, then $(n)_k$ have no repeated date, so

$$
\mathbb P(\text{at least one collision})
=
1-\frac{(n)_k}{n^k}.
$$

For $k>n$, the pigeonhole principle makes a collision certain.


In [ ]:
birthday_n = widgets.IntSlider(value=365, min=2, max=500, description="dates n")
birthday_k = widgets.IntSlider(value=23, min=0, max=100, description="people k")
birthday_output = widgets.Output()

def update_birthday(*_):
    with birthday_output:
        clear_output(wait=True)
        n, k = birthday_n.value, birthday_k.value
        p = birthday_collision_probability(k, n)
        if k <= n:
            display(Math(
                rf"\mathbb P(\mathrm{{collision}})"
                rf"=1-\frac{{({n})_{{{k}}}}}{{{n}^{{{k}}}}}"
                rf"\approx {p:.6f}"
            ))
        else:
            display(Math(rf"k={k}>n={n}\Longrightarrow\mathbb P(\mathrm{{collision}})=1"))

        maximum = min(100, n + 1)
        xs = np.arange(0, maximum + 1)
        ys = [birthday_collision_probability(int(x), n) for x in xs]
        fig, ax = plt.subplots(figsize=(8, 3.3))
        ax.plot(xs, ys)
        ax.axhline(0.5, linestyle="--")
        ax.axvline(min(k, maximum), linestyle=":")
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlabel("number of individuals k")
        ax.set_ylabel("collision probability")
        ax.set_title(rf"Birthday-type collisions with $n={n}$ possible dates")
        plt.show()

for c in (birthday_n, birthday_k):
    c.observe(update_birthday, names="value")
display(widgets.VBox([widgets.HBox([birthday_n, birthday_k]), birthday_output]))
update_birthday()


## 13. Recurring counting patterns

The same formulas recur because the same **outcome structures** recur.

| Structure | Natural count |
|---|---:|
| $k$ recorded positions, $n$ reusable types | $n^k$ |
| $k$ recorded positions, no object reused | $(n)_k$ |
| $k$-element subset of $n$ distinct objects | $\binom nk$ |
| fixed type totals $n_1,\ldots,n_m$ in an ordered sequence | $\binom{n}{n_1,\ldots,n_m}$ |
| unordered reusable type counts summing to $k$ | $\binom{n+k-1}{k}$ |
| at least one of several overlapping conditions | inclusion--exclusion |

> **Key modelling lesson:** there is no correct counting formula until the elementary outcome has been specified precisely.


## 14. Guided exercise generator

Press **New exercise**, solve on paper first, and then use the hint/check/reveal controls.  
The generator deliberately mixes structural decisions with arithmetic.


In [ ]:
exercise_rng = random.Random(20260815)
exercise_type = widgets.Dropdown(
    options=[
        ("Random type", "random"),
        ("Set operation", "set"),
        ("Pigeonhole", "pigeon"),
        ("Committee", "committee"),
        ("Stars and bars", "stars"),
        ("Sampling scheme", "sampling"),
        ("Circular arrangement", "circular"),
    ],
    value="random",
    description="Type",
)
new_exercise_button = widgets.Button(description="New exercise")
hint_button = widgets.Button(description="Hint")
reveal_button = widgets.Button(description="Reveal solution")
check_button = widgets.Button(description="Check answer")
exercise_answer = widgets.Text(description="Answer")
exercise_prompt_output = widgets.Output()
exercise_feedback_output = widgets.Output()
exercise_state = {}

def make_exercise(_=None):
    chosen = exercise_type.value
    if chosen == "random":
        chosen = exercise_rng.choice(["set", "pigeon", "committee", "stars", "sampling", "circular"])

    if chosen == "set":
        U = set(range(1, 9))
        A = set(exercise_rng.sample(list(U), 4))
        B = set(exercise_rng.sample(list(U), 4))
        ans = len(A ^ B)
        prompt = f"Let Ω={{1,…,8}}, A={sorted(A)}, B={sorted(B)}. Find |A △ B|."
        hint = r"Use \(|A\triangle B|=|A|+|B|-2|A\cap B|\)."
        solution = rf"|A\triangle B|={len(A)}+{len(B)}-2({len(A&B)})={ans}."
    elif chosen == "pigeon":
        N = exercise_rng.randint(20, 70)
        m = exercise_rng.randint(3, 9)
        ans = math.ceil(N / m)
        prompt = f"{N} objects are placed into {m} boxes. What minimum occupancy is guaranteed for at least one box?"
        hint = r"Use the generalized pigeonhole principle."
        solution = rf"\left\lceil {N}/{m}\right\rceil={ans}."
    elif chosen == "committee":
        n = exercise_rng.randint(8, 15)
        k = exercise_rng.randint(2, min(6, n))
        ans = math.comb(n, k)
        prompt = f"How many unordered committees of size {k} can be chosen from {n} distinct people?"
        hint = "Order does not matter and nobody is selected twice."
        solution = rf"\binom{{{n}}}{{{k}}}={ans}."
    elif chosen == "stars":
        r = exercise_rng.randint(5, 12)
        m = exercise_rng.randint(2, 5)
        ans = math.comb(r + m - 1, m - 1)
        prompt = f"How many non-negative integer solutions satisfy x₁+⋯+x_{m}={r}?"
        hint = "Use stars and bars."
        solution = rf"\binom{{{r+m-1}}}{{{m-1}}}={ans}."
    elif chosen == "sampling":
        n = exercise_rng.randint(5, 10)
        k = exercise_rng.randint(2, min(4, n))
        ordered = exercise_rng.choice([True, False])
        replace = exercise_rng.choice([True, False])
        if ordered and replace:
            ans, formula = n**k, rf"{n}^{k}"
            description = "ordered, with replacement"
        elif ordered and not replace:
            ans, formula = falling_factorial(n, k), rf"({n})_{k}"
            description = "ordered, without replacement"
        elif not ordered and replace:
            ans, formula = math.comb(n+k-1, k), rf"\binom{{{n+k-1}}}{{{k}}}"
            description = "unordered type-count sample, with replacement"
        else:
            ans, formula = math.comb(n, k), rf"\binom{{{n}}}{{{k}}}"
            description = "unordered, without replacement"
        prompt = f"Count a sample of size {k} from {n} available objects/types: {description}."
        hint = "Classify the experiment in the 2×2 sampling table."
        solution = rf"{formula}={ans}."
    else:
        n = exercise_rng.randint(4, 9)
        ans = math.factorial(n - 1)
        prompt = f"How many circular arrangements of {n} distinct people are there if rotations are identical?"
        hint = "Each circular arrangement corresponds to n linear rotations."
        solution = rf"({n}-1)!={ans}."

    exercise_state.clear()
    exercise_state.update(answer=ans, hint=hint, solution=solution)
    exercise_answer.value = ""
    with exercise_prompt_output:
        clear_output(wait=True)
        display(Markdown("### Exercise\n" + prompt))
    with exercise_feedback_output:
        clear_output(wait=True)

def show_hint(_):
    with exercise_feedback_output:
        clear_output(wait=True)
        display(Markdown("**Hint:** " + exercise_state.get("hint", "")))

def reveal_solution(_):
    with exercise_feedback_output:
        clear_output(wait=True)
        display(Markdown("**Solution:**"))
        display(Math(exercise_state.get("solution", "")))

def check_exercise(_):
    with exercise_feedback_output:
        clear_output(wait=True)
        try:
            guess = int(exercise_answer.value.strip())
        except ValueError:
            display(Markdown("Enter an integer answer."))
            return
        if guess == exercise_state.get("answer"):
            display(Markdown("**Correct.** Now explain *why* the formula counts every outcome exactly once."))
        else:
            display(Markdown("**Not yet.** Recheck the description of one elementary outcome before recomputing."))

new_exercise_button.on_click(make_exercise)
hint_button.on_click(show_hint)
reveal_button.on_click(reveal_solution)
check_button.on_click(check_exercise)

display(widgets.VBox([
    widgets.HBox([exercise_type, new_exercise_button]),
    exercise_prompt_output,
    widgets.HBox([exercise_answer, check_button]),
    widgets.HBox([hint_button, reveal_button]),
    exercise_feedback_output,
]))
make_exercise()


## 15. AI Audit

AI can produce a numerically correct-looking formula for the **wrong sample space**.  
Use it as a conjecture generator, then audit the reasoning.

### Audit protocol

For any AI solution, check:

1. **Outcome:** What exactly is one elementary outcome?
2. **Order:** Is order recorded?
3. **Replacement:** Can an object/type appear again?
4. **Distinguishability:** Are objects individually labelled or only their types recorded?
5. **Overcount:** Does the argument count one visible outcome several times?
6. **Small case:** Can the claim be checked by enumeration for small $n,k$?
7. **Boundary cases:** Does it behave correctly at $k=0$, $k=n$, or $k>n$?

### Suggested AI audit prompts

- “Explain why unordered sampling with replacement gives $\binom{n+k-1}{k}$. Then verify $n=3,k=2$ by listing all outcomes.”
- “Solve Galileo's three-dice problem. State explicitly whether the elementary outcomes are ordered or unordered.”
- “Give a counting proof of $|A\triangle B|=|A|+|B|-2|A\cap B|$, and test it on finite sets.”
- “Derive the hypergeometric probability from favourable and total sample counts; identify the feasible support of $k$.”

> A good audit does not ask only **“Is the final number correct?”** It asks **“Is the model and counting argument correct?”**


## 16. Self-check quiz

Choose one answer for each question and press **Grade quiz**.  
The questions emphasize structural recognition rather than arithmetic alone.


In [ ]:
quiz_data = [
    (
        "1. If |A|=7, |B|=6 and |A∩B|=2, then |A△B| is:",
        ["Choose...", "9", "11", "13", "15"],
        "9",
        r"|A\triangle B|=7+6-2(2)=9",
    ),
    (
        "2. Ordered samples of length 3 from 8 types, with replacement:",
        ["Choose...", "56", "336", "512", "720"],
        "512",
        r"8^3=512",
    ),
    (
        "3. Four-element subsets of a ten-element set:",
        ["Choose...", "40", "210", "5040", "10000"],
        "210",
        r"\binom{10}{4}=210",
    ),
    (
        "4. Non-negative solutions of x₁+x₂+x₃=7:",
        ["Choose...", "21", "28", "36", "45"],
        "36",
        r"\binom{9}{2}=36",
    ),
    (
        "5. Circular arrangements of 6 distinct people (rotations identical):",
        ["Choose...", "60", "120", "360", "720"],
        "120",
        r"(6-1)!=120",
    ),
    (
        "6. If 23 objects are placed in 5 boxes, some box contains at least:",
        ["Choose...", "4", "5", "6", "23"],
        "5",
        r"\lceil 23/5\rceil=5",
    ),
    (
        "7. In the birthday model, if k>n, collision probability is:",
        ["Choose...", "0", "1/2", "(n)_k/n^k", "1"],
        "1",
        r"k>n\Longrightarrow \mathbb P(\mathrm{collision})=1",
    ),
    (
        "8. Galileo's three-dice sample space has:",
        ["Choose...", "18", "56", "216", "729"],
        "216",
        r"6^3=216\text{ ordered triples}",
    ),
]

quiz_widgets = []
quiz_rows = []
for prompt, options, _, _ in quiz_data:
    dropdown = widgets.Dropdown(options=options, value="Choose...", layout=widgets.Layout(width="190px"))
    quiz_widgets.append(dropdown)
    quiz_rows.append(widgets.HBox([
        widgets.HTML(f"<div style='width:650px'>{prompt}</div>"),
        dropdown,
    ]))

grade_button = widgets.Button(description="Grade quiz")
quiz_output = widgets.Output()

def grade_quiz(_):
    with quiz_output:
        clear_output(wait=True)
        score = 0
        for widget, (_, _, correct, explanation) in zip(quiz_widgets, quiz_data):
            if widget.value == correct:
                score += 1
        display(Markdown(f"### Score: {score}/{len(quiz_data)}"))
        for i, (widget, (_, _, correct, explanation)) in enumerate(zip(quiz_widgets, quiz_data), 1):
            mark = "✓" if widget.value == correct else "✗"
            display(Markdown(f"**{mark} Question {i}:** correct answer = `{correct}`"))
            display(Math(explanation))

grade_button.on_click(grade_quiz)
display(widgets.VBox(quiz_rows + [grade_button, quiz_output]))


## 17. Automatic mathematical verification

These checks do not replace proofs. They verify that the notebook's implementations agree with exact combinatorial identities and small exhaustive enumerations.


In [ ]:
# Basic set identities
U = set(range(10))
A = {1, 2, 3, 7}
B = {3, 4, 7, 8}
assert A ^ B == (A - B) | (B - A)
assert A ^ B == (A | B) - (A & B)
assert len(A ^ B) == len(A) + len(B) - 2 * len(A & B)

# Power-set size
for n in range(8):
    base = list(range(n))
    count = sum(len(list(combinations(base, r))) for r in range(n + 1))
    assert count == 2 ** n

# Falling factorial and ordered enumeration
for n in range(1, 8):
    for k in range(n + 1):
        assert falling_factorial(n, k) == len(list(permutations(range(n), k)))

# Combination counts
for n in range(1, 9):
    for k in range(n + 1):
        assert math.comb(n, k) == len(list(combinations(range(n), k)))

# Pascal identity
for n in range(1, 12):
    for k in range(1, n + 1):
        assert math.comb(n + 1, k) == math.comb(n, k) + math.comb(n, k - 1)

# Vandermonde identity
for r in range(1, 7):
    for s in range(1, 7):
        for n in range(r + s + 1):
            lhs = sum(
                math.comb(r, k) * math.comb(s, n - k)
                for k in range(n + 1)
                if k <= r and n - k <= s
            )
            assert lhs == math.comb(r + s, n)

# Stars and bars against enumeration
for r in range(7):
    for m in range(1, 5):
        assert len(compositions(r, m, 0)) == math.comb(r + m - 1, m - 1)

# Derangement benchmarks
assert [derangement(n) for n in range(7)] == [1, 0, 1, 2, 9, 44, 265]

# Hypergeometric normalization
N, K, sample_n = 30, 9, 7
support = range(max(0, sample_n - (N - K)), min(sample_n, K) + 1)
probability_sum = sum(
    math.comb(K, k) * math.comb(N - K, sample_n - k) / math.comb(N, sample_n)
    for k in support
)
assert abs(probability_sum - 1.0) < 1e-12

# Galileo's counts
dice = list(product(range(1, 7), repeat=3))
assert sum(sum(w) == 9 for w in dice) == 25
assert sum(sum(w) == 10 for w in dice) == 27

show_math_result(
    "All automatic checks passed",
    r"|A\triangle B|=|A|+|B|-2|A\cap B|",
    r"\sum_k\mathbb P(X=k)=1",
    r"N_9=25,\qquad N_{10}=27",
    note="Set identities, sampling counts, Pascal/Vandermonde, stars and bars, "
         "derangements, hypergeometric normalization and Galileo's counts all passed."
)


## Further work

To use this notebook as a learning tool rather than a calculator, repeat the following workflow:

1. describe the elementary outcomes in words;
2. decide whether order matters;
3. decide whether repetition/replacement is allowed;
4. decide whether the objects are distinguishable;
5. predict the formula before running code;
6. verify a small case by enumeration when possible;
7. explain why every outcome is counted exactly once;
8. only then use the general formula.

The notebook is self-contained and can be opened directly in Jupyter or Google Colab.
